# Eyeballing the `expected_words` baseline

Sample a board, show the clue the model gives, the own words it intends
the guesser to find, and the nearest dangerous words it is betting
against. The point is to judge clue *quality* by eye, which no aggregate
statistic does.

The row that matters most is **rank** in the last table. A guesser works
down its own similarity ranking and stops at its first mistake, so a clue
is only as good as its worst intended word's position: if any non-own
word outranks the k-th intended word, the turn is likely to break there.

Boards are built from `load_holdout_wordlist()` — the 150 words held out
of training — so these are the same kind of boards the frozen eval suite
uses. Requires `cache/similarity_tensor.npy` and `cache/clue_stats.npz`
(`scripts/data/build_similarity_tensor.py`, `scripts/data/build_clue_stats.py`).

In [ ]:
import sys
from pathlib import Path

# Notebook lives in scratch/, one level below the project root.
PROJECT_ROOT = Path.cwd().parent if (Path.cwd().parent / "codenames").exists() else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import random

import numpy as np

from codenames.board import Board, Role, load_holdout_wordlist, load_wordlist
from codenames.clue_stats import ClueStats
from codenames.similarity import SimilarityTensor
from codenames.spymasters.base import TurnContext
from codenames.spymasters.registry import spymaster_spec

sims = SimilarityTensor.load()
stats = ClueStats.load()

SPYMASTER_CLS, SPYMASTER_KWARGS = spymaster_spec("expected_words")
spymaster = SPYMASTER_CLS(**SPYMASTER_KWARGS)
SPACE = SPYMASTER_KWARGS.get("space", "numberbatch")
print(f"{type(spymaster).__name__}  space={SPACE}  params={SPYMASTER_KWARGS}")

In [ ]:
ROLE_TAG = {Role.OWN: "own", Role.OPPONENT: "OPPONENT", Role.NEUTRAL: "neutral", Role.ASSASSIN: "ASSASSIN"}


def z_of(clue: str, words: list[str], space: str = SPACE) -> dict[str, float]:
    """Per-clue z-score of `clue` against each board word -- the same
    quantity the spymaster thresholds on, so what's printed below is
    literally what it saw."""
    ci = sims.clue_index[clue.lower()]
    si = stats.space_index(space)
    mu, sd = float(stats.mean[ci, si]), float(stats.std[ci, si])
    bi = sims.board_index
    return {w: (float(sims.tensor[ci, bi[w.lower()], si]) - mu) / sd for w in words}


def sample_board(seed: int | None = None, n_revealed: int = 0, holdout_only: bool = True) -> Board:
    """A board, optionally part-played. `n_revealed` reveals that many
    random non-assassin cards -- late-game states are where the model
    behaves differently (fewer own words left to cover), so it's worth
    being able to look at them."""
    seed = random.randrange(1_000_000) if seed is None else seed
    vocab = load_holdout_wordlist() if holdout_only else load_wordlist()
    board = Board.generate(seed, vocabulary=vocab)
    if n_revealed:
        rng = random.Random(seed ^ 0x5EED)
        safe = [w for w in board.words if board.role_of(w) is not Role.ASSASSIN]
        # Always leave at least one own word unrevealed -- a board with none
        # is a finished game, and asking for a clue there is meaningless.
        own = [w for w in safe if board.role_of(w) is Role.OWN]
        keep = rng.choice(own)
        for w in rng.sample([s for s in safe if s != keep], min(n_revealed, len(safe) - 1)):
            board.reveal(w)
    return board

In [ ]:
def inspect(seed: int | None = None, n_revealed: int = 0, alternatives: int = 3, ranking: int = 10,
            holdout_only: bool = True) -> None:
    """Print one board's clue and everything needed to judge it by eye."""
    board = sample_board(seed, n_revealed, holdout_only)
    ctx = TurnContext(board=board, turn_index=len(board.revealed))
    picks = spymaster.top_clues(ctx, sims, alternatives)
    clue, number, score = picks[0]

    unrevealed = [w for w in board.words if not board.is_revealed(w)]
    z = z_of(clue, unrevealed)
    by_role = {r: sorted((w for w in unrevealed if board.role_of(w) is r), key=lambda w: -z[w])
               for r in (Role.OWN, Role.OPPONENT, Role.NEUTRAL, Role.ASSASSIN)}

    intended = by_role[Role.OWN][:number]           # the model intends the top-k own words
    weakest = z[intended[-1]] if intended else float("nan")
    non_own = [w for w in unrevealed if board.role_of(w) is not Role.OWN]
    nearest = max(non_own, key=lambda w: z[w]) if non_own else None

    print(f"board {board.seed}" + (f"  ({len(board.revealed)} revealed)" if board.revealed else "")
          + f"   CLUE: {clue.upper()}  {number}      score={score:.3f}")
    print(f"\n  intends ({number}):")
    for w in intended:
        print(f"    {w:<16s} z={z[w]:+.2f}")

    print("\n  nearest word of each other role:")
    for role in (Role.OPPONENT, Role.NEUTRAL, Role.ASSASSIN):
        words = by_role[role]
        if not words:
            print(f"    {ROLE_TAG[role]:<9s} -- none left --")
            continue
        w = words[0]
        print(f"    {ROLE_TAG[role]:<9s} {w:<16s} z={z[w]:+.2f}   margin below weakest intended: {weakest - z[w]:+.2f}")

    if nearest is not None:
        verdict = "CLEAR" if z[nearest] < weakest else "*** OUTRANKED ***"
        print(f"\n  weakest intended {weakest:+.2f}  vs  nearest distractor {z[nearest]:+.2f}   -> {verdict}")

    print(f"\n  ranking a similarity-ranking guesser would work down:")
    for i, w in enumerate(sorted(unrevealed, key=lambda w: -z[w])[:ranking], start=1):
        mark = "<-- intended" if w in intended else ""
        print(f"    {i:>2}. {w:<16s} {ROLE_TAG[board.role_of(w)]:<9s} z={z[w]:+.2f}  {mark}")

    if alternatives > 1:
        print("\n  runners-up:")
        for c, n, s in picks[1:]:
            print(f"    {c:<16s} {n}   score={s:.3f}")
    print()

In [ ]:
inspect(seed=7)

In [ ]:
# A few fresh boards. Call inspect() with no seed for a new random one each time.
for s in (101, 102, 103):
    inspect(seed=s, ranking=6, alternatives=1)

In [ ]:
# Part-played board: fewer own words left, so the model claims fewer.
inspect(seed=7, n_revealed=12, ranking=8, alternatives=1)

In [ ]:
def sweep(n: int = 25, n_revealed: int = 0) -> None:
    """One line per board -- for skimming many clues quickly. `outranked`
    flags any board where a non-own word beats the weakest intended word,
    which is the failure this whole metric exists to avoid."""
    print(f"{'seed':>6} {'clue':<16} {'n':>2}  {'weakest':>8} {'nearest':>8} {'margin':>7}  worst distractor")
    bad = 0
    for seed in range(n):
        board = sample_board(seed, n_revealed)
        ctx = TurnContext(board=board, turn_index=len(board.revealed))
        clue, number, _ = spymaster.top_clues(ctx, sims, 1)[0]
        unrevealed = [w for w in board.words if not board.is_revealed(w)]
        z = z_of(clue, unrevealed)
        own = sorted((w for w in unrevealed if board.role_of(w) is Role.OWN), key=lambda w: -z[w])
        non_own = [w for w in unrevealed if board.role_of(w) is not Role.OWN]
        weakest = z[own[number - 1]]
        nearest = max(non_own, key=lambda w: z[w])
        flag = "  <-- OUTRANKED" if z[nearest] >= weakest else ""
        bad += bool(flag)
        print(f"{board.seed:>6} {clue:<16} {number:>2}  {weakest:>+8.2f} {z[nearest]:>+8.2f} "
              f"{weakest - z[nearest]:>+7.2f}  {nearest} ({ROLE_TAG[board.role_of(nearest)]}){flag}")
    print(f"\nboards where a non-own word outranks the weakest intended word: {bad}/{n}")


sweep(25)